# 単語単位のラベルをサブワードに揃える (word_ids)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/06_subword/word_ids.ipynb)

NLP クックブック 06-10 ／ サブワードとトークナイザ

単語ごとに付けたラベル (品詞・固有表現タグ) を、サブワード分割後のトークン列に対応させたい。

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
words = ["Hanako", "Yamada", "works", "at", "Hokkaido", "University", "."]
labels = ["B-PER", "I-PER", "O", "O", "B-ORG", "I-ORG", "O"]

enc = tokenizer(words, is_split_into_words=True)
print(enc.tokens())
print(enc.word_ids())

['[CLS]', 'Han', '##ak', '##o', 'Ya', '##mada', 'works', 'at', 'Ho', '##kka', '##ido', 'University', '.', '[SEP]']
[None, 0, 0, 0, 1, 1, 2, 3, 4, 4, 4, 5, 6, None]


In [2]:
# 各単語の先頭サブワードにだけラベルを付け、残りと特殊トークンは -100 (損失計算で無視される)
label2id = {l: i for i, l in enumerate(sorted(set(labels)))}
aligned, prev = [], None
for wid in enc.word_ids():
    if wid is None or wid == prev:
        aligned.append(-100)
    else:
        aligned.append(label2id[labels[wid]])
    prev = wid
for tok, lab in zip(enc.tokens(), aligned):
    print(f"{tok:12}{lab}")

[CLS]       -100
Han         1
##ak        -100
##o         -100
Ya          3
##mada      -100
works       4
at          4
Ho          0
##kka       -100
##ido       -100
University  2
.           4
[SEP]       -100


- 予測時も同じ `word_ids()` を使って、各単語の先頭サブワードの予測だけを取り出す。